# aerial-vg: TransVG on DIOR-RSVG + direction-aware D4 augmentation

**Settings:** Accelerator = **GPU T4 x2**, Internet = **On**. The dataset is downloaded straight from the authors' Google Drive into `/tmp` (no upload needed; ~5 GB, a few minutes).

Runs 4 conditions in 2 parallel phases (one per GPU):
| name | `--geo_aug` | role |
|---|---|---|
| baseline | hflip | TransVG default (reproduction) |
| d4_remap | d4_remap | proposed |
| d4_noremap | d4_noremap | control: same geometry, text not remapped |
| no_geo | none | ablation: no geometric augmentation |

Use **Save Version → Save & Run All (Commit)** so it keeps running after you close the browser (12 h limit).

In [ ]:
REPO_URL = 'https://github.com/HonourGok/aerial-vg.git'   # used only if the code is not attached as a Kaggle dataset
import os, subprocess, glob, shutil
DATA_ROOT = '/tmp/DIOR_RSVG'   # /tmp is not saved as notebook output, so the 5 GB stay out of the 20 GB output limit

def find_dir(root, ext):
    best = None
    for d, _, files in os.walk(root):
        n = sum(f.lower().endswith(ext) for f in files)
        if n and (best is None or n > best[1]):
            best = (d, n)
    return best

if not os.path.isdir(DATA_ROOT + '/Annotations'):
    # 1) a manually uploaded Kaggle dataset, if one was added as input
    up = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/Annotations', recursive=True)]
    if up:
        DATA_ROOT = up[0]
    else:
        # 2) download from the authors' Google Drive folder (Annotations.zip 7 MB, JPEGImages.zip 5 GB)
        subprocess.run('pip -q install gdown', shell=True, check=True)
        os.makedirs('/tmp/rsvg_dl', exist_ok=True)
        subprocess.run('gdown --folder https://drive.google.com/drive/folders/1hTqtYsC6B-m4ED2ewx5oKuYZV13EoJp_ -O /tmp/rsvg_dl',
                       shell=True, check=True)
        os.makedirs(DATA_ROOT, exist_ok=True)
        for z in glob.glob('/tmp/rsvg_dl/**/*.zip', recursive=True):
            subprocess.run(f'unzip -q -o "{z}" -d /tmp/rsvg_unz', shell=True, check=True)
            os.remove(z)
        for sub, ext in (('Annotations', '.xml'), ('JPEGImages', '.jpg')):
            d, n = find_dir('/tmp/rsvg_unz', ext)
            print(sub, '<-', d, n, 'files')
            shutil.move(d, f'{DATA_ROOT}/{sub}')

print('DATA_ROOT =', DATA_ROOT, '| xml:', len(os.listdir(DATA_ROOT + '/Annotations')),
      '| jpg:', len(os.listdir(DATA_ROOT + '/JPEGImages')))
# If gdown fails with a quota / "too many users" error: download the two zips manually and upload them as a
# Kaggle dataset (Datasets -> New Dataset), add it as input, and re-run this cell.

In [ ]:
%cd /kaggle/working
# code: from an attached Kaggle dataset containing aerial-vg/ (works while the GitHub repo is private), else git clone
code_src = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/aerial-vg/train.py', recursive=True)]
!rm -rf aerial-vg
if code_src:
    shutil.copytree(code_src[0], '/kaggle/working/aerial-vg')
    print('code from', code_src[0])
else:
    subprocess.run(f'git clone -q {REPO_URL} aerial-vg', shell=True, check=True)
%cd /kaggle/working/aerial-vg
!(git rev-parse HEAD 2>/dev/null || md5sum train.py datasets/geo_aug.py) | tee results_commit.txt
!pip -q install -r requirements.txt
!mkdir -p checkpoints && wget -q -nc -O checkpoints/detr-r50-e632da11.pth https://dl.fbaipublicfiles.com/detr/detr-r50-e632da11.pth
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import torch,torchvision,transformers;print(torch.__version__,torchvision.__version__,transformers.__version__)" 

In [ ]:
os.environ['DATA_ROOT'] = DATA_ROOT
os.environ['DETR'] = '/kaggle/working/aerial-vg/checkpoints/detr-r50-e632da11.pth'
# compute budget (identical for all conditions)
os.environ.update(EPOCHS='15', LR_DROP='10', FRACTION='0.5', BS='8', IMSIZE='640', MAX_HOURS='4.8', SEED='13')
!python tools/dataset_stats.py --data_root $DATA_ROOT --split_root splits --out results/dataset_stats.json

In [ ]:
# Quick check (~2 min): one tiny run to catch errors before the long jobs. Delete afterwards.
!EPOCHS=1 FRACTION=0.01 MAX_HOURS=1 bash scripts/run_one.sh _smoke hflip 0 2>&1 | tail -n 25
!rm -rf outputs/_smoke results/_smoke

In [ ]:
# Phase 1: baseline (GPU0) || proposed (GPU1)
!mkdir -p logs
!(bash scripts/run_one.sh baseline hflip 0 > logs/baseline.log 2>&1 &) ; (bash scripts/run_one.sh d4_remap d4_remap 1 > logs/d4_remap.log 2>&1 &)
import time
def wait(names):
    while True:
        done = [os.path.exists(f'results/{n}/metrics.json') for n in names]
        busy = subprocess.run("pgrep -f 'train.py|eval_dior.py'", shell=True, capture_output=True).stdout.strip()
        if all(done) or not busy:
            break
        time.sleep(120)
    for n in names:
        print(n, 'OK' if os.path.exists(f'results/{n}/metrics.json') else 'FAILED - see logs/%s.log' % n)
time.sleep(60); wait(['baseline', 'd4_remap'])
!tail -n 3 logs/baseline.log logs/d4_remap.log

In [ ]:
# Phase 2: control (GPU0) || ablation without geometric augmentation (GPU1)
!(bash scripts/run_one.sh d4_noremap d4_noremap 0 > logs/d4_noremap.log 2>&1 &) ; (bash scripts/run_one.sh no_geo none 1 > logs/no_geo.log 2>&1 &)
time.sleep(60); wait(['d4_noremap', 'no_geo'])

In [ ]:
!python compare.py --runs baseline=results/baseline d4_remap=results/d4_remap d4_noremap=results/d4_noremap no_geo=results/no_geo --ref baseline --out results/comparison
!python tools/viz_examples.py --a results/baseline --b results/d4_remap --data_root $DATA_ROOT --out results/figures/baseline_vs_d4remap.png
!for n in baseline d4_remap d4_noremap no_geo; do cp outputs/$n/log.txt results/$n/train_log.txt; done
!cp logs/*.log results/ ; cp results_commit.txt results/
!cd /kaggle/working && zip -qr results.zip aerial-vg/results && ls -lh results.zip